# 02 — Download daily stock prices

**Goal:** daily prices for every ticker plus the market benchmark (SPY), from Jan 2024 to the end of April 2025. That covers the market-model estimation window (trading days −130 to −11) and the event windows.

| | |
|---|---|
| **Inputs** | Yahoo Finance via `yfinance` (free, no key) |
| **Outputs** | `data/raw/market_data/daily_prices.csv` in long format: one row per (date, ticker) with `close`, `adj_close`, `volume` |

**Why both close and adjusted close?** Returns use `adj_close` (it accounts for dividends and splits). Market cap uses the raw `close`, because shares outstanding from the 10-K are in raw, unadjusted shares.

*Known limitation (document in the README):* Yahoo data can be revised and has no delisted firms. CRSP would be the gold standard. This matters less here because every sample firm traded through April 2025.

In [1]:
# --- Setup: find the repo root and load shared settings from code/config.py ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Can't find the repo root. Run this from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))

import config as cfg
print("Repo root:", ROOT)

Repo root: /content/project


In [2]:
import pandas as pd
import yfinance as yf

cfg.MARKET_DIR.mkdir(parents=True, exist_ok=True)
symbols = cfg.TICKERS + [cfg.MARKET]

# auto_adjust=False so that yfinance returns BOTH "Close" and "Adj Close"
raw = yf.download(symbols, start=cfg.PRICE_START, end=cfg.PRICE_END,
                  auto_adjust=False, progress=False, group_by="column")
raw.tail(3)

Price        Adj Close                                                 \
Ticker            AAPL        BBY         CAT        COST         JNJ   
Date                                                                    
2025-04-25  207.842697  62.597801  301.889587  967.455444  148.796036   
2025-04-28  208.696808  62.385174  302.490540  968.821716  149.537231   
2025-04-29  209.759445  61.645569  302.825439  981.851074  150.076263   

Price                                                                 ...  \
Ticker            MSFT        NKE         SPY        TGT        TSLA  ...   
Date                                                                  ...   
2025-04-25  387.303436  55.126068  541.655334  90.152817  284.950012  ...   
2025-04-28  386.621429  54.829479  541.861877  89.956795  285.880005  ...   
2025-04-29  389.468018  55.049530  545.275269  90.460861  292.029999  ...   

Price        Volume                                                           \
Ticker          BBY      CAT     COST      JNJ      MSFT       NKE       SPY   
Date                                                                           
2025-04-25  5171700  2162400  1780900  8643600  18973200  17156000  61119600   
2025-04-28  3462100  2189600  1616200  5815200  16579400  13321200  47613800   
2025-04-29  2453200  2651000  1523100  5246700  14974000  12968000  47775100   

Price                                     
Ticker          TGT       TSLA       XOM  
Date                                      
2025-04-25  5274900  167560700  11245800  
2025-04-28  4662400  151731800  13890900  
2025-04-29  3768400  108906600   9483100  

[3 rows x 66 columns]

## Reshape to long format (one row per date and ticker)

In [3]:
pieces = []
for field, name in [("Close", "close"), ("Adj Close", "adj_close"), ("Volume", "volume")]:
    wide = raw[field].copy()
    wide.index.name = "date"
    pieces.append(wide.reset_index().melt(id_vars="date", var_name="ticker", value_name=name))

prices = pieces[0].merge(pieces[1], on=["date", "ticker"]).merge(pieces[2], on=["date", "ticker"])
prices = prices.dropna(subset=["adj_close"]).sort_values(["ticker", "date"]).reset_index(drop=True)
prices.to_csv(cfg.PRICES_CSV, index=False)
print(f"Saved {len(prices):,} rows to {cfg.PRICES_CSV.relative_to(cfg.ROOT)}")

# Provenance: record when and how the prices were pulled (this small file IS committed to git)
from datetime import datetime, timezone
cfg.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    "source": "Yahoo Finance via yfinance", "yfinance_version": getattr(yf, "__version__", "unknown"),
    "symbols": " ".join(symbols), "start": cfg.PRICE_START, "end_exclusive": cfg.PRICE_END,
    "rows": len(prices), "downloaded_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
}]).to_csv(cfg.PROCESSED_DIR / "prices_download_log.csv", index=False)

Saved 3,652 rows to data/raw/market_data/daily_prices.csv


## Checks

In [4]:
coverage = prices.groupby("ticker")["date"].agg(["min", "max", "count"])
print(coverage)

missing = sorted(set(symbols) - set(prices["ticker"]))
if missing:
    # Log the problem instead of failing silently
    probs = pd.read_csv(cfg.DOWNLOAD_PROBLEMS_CSV)
    new = pd.DataFrame({"ticker": missing, "step": "02_prices", "problem": "no price data from yfinance"})
    pd.concat([probs, new]).to_csv(cfg.DOWNLOAD_PROBLEMS_CSV, index=False)
    print("WARNING: no prices for", missing)

assert cfg.MARKET in set(prices["ticker"]), "market benchmark missing: cannot compute abnormal returns"
assert pd.Timestamp(cfg.EVENT_DAY0) in set(pd.to_datetime(prices["date"])), "event day 0 missing from price data"
assert not prices.duplicated(["date", "ticker"]).any(), "duplicate (date, ticker) rows"

              min        max  count
ticker                             
AAPL   2024-01-02 2025-04-29    332
BBY    2024-01-02 2025-04-29    332
CAT    2024-01-02 2025-04-29    332
COST   2024-01-02 2025-04-29    332
JNJ    2024-01-02 2025-04-29    332
MSFT   2024-01-02 2025-04-29    332
NKE    2024-01-02 2025-04-29    332
SPY    2024-01-02 2025-04-29    332
TGT    2024-01-02 2025-04-29    332
TSLA   2024-01-02 2025-04-29    332
XOM    2024-01-02 2025-04-29    332
